### COMBOを用いたベイズ最適化

東京大学津田宏治グループで開発されたベイズ最適化のためのpythonソフトです。(参考文献[2])
windows用のスタンドアロンソフトも存在します。(参考文献[3])

ubuntuでpipを用いる場合のinstallは以下になります．
1. fortran compilerが必要です．まず，gfortranなどをinstallしてください．

```
sudo apt install gfortran
```

2. cythonのinstall

```
sudo apt install cython
```
が必要かもしれません．
次に
```
pip install cython
```

3. combo3のinstall

```
git clone https://github.com/tsudalab/combo3.git
cd combo3
python setup.py install
```

In [ ]:
import pandas as pd
import random
from sklearn import preprocessing
from sklearn.gaussian_process import GaussianProcessRegressor, kernels
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
import combo


以下でcombo用のinterfaceを定義します。

In [ ]:
def load_data():
    """load data

    Returns:
        np.array: descriptor
        np.array: target values
    """
    df = pd.read_csv(
        "../data_calculated/Carbon8_descriptor_energy.csv", index_col=[0])
    descriptor_names = ['a0.25_rp1.5', 'a0.25_rp2.5', 'a0.5_rp1.5', 'a0.5_rp2.5',
                        'a1.0_rp1.5', 'a1.0_rp2.5']
    target_name = 'minus_energy'
    X = df[descriptor_names].values
    y = df[target_name].values
    return X, y


X, t = load_data()


In [ ]:
class simulator:
    """necessary to define simulator class
    """

    def __init__(self):
        """initialization. loading data
        """
        _, self.t = load_data()

    def __call__(self, action):
        """__call__

        Args:
            action (int): action index

        Returns:
            float: target value to action
        """
        print("call simiulator action=", action)
        return self.t[action]


In [ ]:
policy = None
policy = combo.search.discrete.policy(test_X=X)
seed = 1
policy.set_seed(seed)


In [ ]:
res = None
print("warning up")
res = policy.random_search(max_num_probes=10, simulator=simulator())


In [ ]:
# res = policy.bayes_search(max_num_probes=60, simulator=simulator(), score='PI',
#                                      interval=20, num_rand_basis=5000 )
# でも良いが途中で止まってくれないので20回毎繰り返す。

# 以下を３回くりかえす。
for i in range(3):
    # score = EI, PI, TS
    # max_num_probes 20回行う。interval=0最初だけparameter最適化を行う。
    res = policy.bayes_search(max_num_probes=20, simulator=simulator(), score='PI',
                              interval=0, num_rand_basis=5000)
    if 0 in res.chosed_actions[:res.total_num_search]:
        break


In [ ]:
best_fx, best_action = res.export_all_sequence_best_fx()
plt.plot(res.fx[:res.total_num_search], "o-", label="chosen")
plt.plot(best_fx, "o-", label="best")
plt.ylabel("$f(x)$")
plt.xlabel("iteration")
plt.legend()
plt.show()

plt.plot(res.chosed_actions[:res.total_num_search], "o-", label="chosen")
plt.plot(best_action, "o-", label="best")
plt.ylabel("action")
plt.xlabel("iteration")
plt.legend()
plt.show()


参考文献 

1. "COMBO: An efficient Bayesian optimization library for materials science"
Tsuyoshi Ueno,
Trevor David Rhone,
Zhufeng Hou,
Teruyasu Mizoguchi,
KojiTsuda, 
http://www.sciencedirect.com/science/article/pii/S2352924516300035

2. COMBO repository
https://github.com/tsudalab/combo

3. "Designing Nanostructures for Phonon Transport via Bayesian Optimization"
Shenghong Ju,
Takuma Shiga,
Lei Feng,
Zhufeng Hou,
Koji Tsuda,
and Junichiro Shiomi
https://journals.aps.org/prx/pdf/10.1103/PhysRevX.7.021024

4. COMBO.exe
https://www.tsudalab.org/project/mitools/

5. "Efficient recommendation tool of materials by executable file based on machine learning", 
Kei Terayama, Koji Tsuda, Ryo Tamura, https://iopscience.iop.org/article/10.7567/1347-4065/ab349b